<a href="https://colab.research.google.com/github/gmauricio-toledo/math-ML/blob/main/Tareas/Tarea-05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1> Tarea 5: Regresión y Clasificación con Machine Learning Clásico

# Parte 1

## Instrucciones

Para cada uno de los problemas asignados (clasificación, regresión y el ejercicio asignado), deberás realizar un ciclo completo de Machine Learning de principio a fin, documentando tus decisiones. Sigue estrictamente estos pasos:

1. **Partición de Datos y Modelo Base (Baseline):**
   Separa tus datos en conjuntos de entrenamiento y prueba. Define y evalúa un modelo base ingenuo ([DummyClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html) para clasificación y [DummyRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyRegressor.html) para regresión) para establecer el baseline que tus modelos están obligados a superar.

2. **Análisis Exploratorio de Datos (EDA):**
   **Utilizando solo el conjunto de entrenamiento**, asegúrate de revisar valores faltantes, tipos de variables, distribución y rangos de las variables, balance de clases (en clasificación) y posibles problemas de multicolinealidad.

3. **Preprocesamiento y Feature Engineering:**
   Implementa las transformaciones dictadas por tu EDA: imputación de faltantes, re-escalamientos y codificación de variables. Además, **es obligatorio que incluyas al menos un método de selección de características o reducción de dimensionalidad en tu experimentación**. Si decides que tu modelo final no debe llevar este paso, deberás justificarlo con evidencia empírica (mostrando que las métricas empeoraron o no justificaron la complejidad extra).

4. **Experimentación Exhaustiva y Pipelines:**
   Se espera una búsqueda de hiperparámetros no trivial. Para cada ejercicio deberás:
   * Probar **al menos tres algoritmos distintos**.
   * Utilizar búsqueda de hiperparámetros (GridSearch o RandomSearch) con un espacio de búsqueda amplio y justificado. No basta con probar un par de configuraciones. Ajusta tu espacio de busqueda de acuerdo a los límites de colab.
   * Para evitar el *data leakage*, **todo tu flujo de preprocesamiento y entrenamiento debe estar encapsulado en un `Pipeline`**.

5. **Evaluación y Selección del Mejor Modelo:**
   La métrica de optimización será **F1-score** para las tareas de clasificación y **MAE** para las de regresión.
   Presenta el rendimiento de tu mejor modelo (el Pipeline final) evaluándolo por primera y única vez sobre tu conjunto de prueba. Presenta también la matriz de confusión (para tareas de clasificación) y el análisis de errores (para tareas de regresión). A continuación se muestra un bloque de código de ejemplo que puedes usar para tu análisis visual de errores en las tareas de regresión (requiere `matplotlib` y `seaborn`):

```python
import matplotlib.pyplot as plt
import seaborn as sns

# Supongamos que 'y_test' son tus valores reales y 'y_pred' tus predicciones
errores = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Gráfico 1: Predicciones vs Valores Reales
# Un modelo perfecto hace que los puntos caigan sobre la línea roja punteada
sns.scatterplot(x=y_test, y=y_pred, ax=axes[0], alpha=0.6)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], '--r', linewidth=2)
axes[0].set_xlabel('Valores Reales')
axes[0].set_ylabel('Predicciones')
axes[0].set_title('Predicciones vs Valores Reales')

# Gráfico 2: Histograma de Errores (Residuos)
# Buscamos una distribución aproximadamente normal y centrada en 0
sns.histplot(errores, kde=True, ax=axes[1])
axes[1].axvline(x=0, color='r', linestyle='--')
axes[1].set_xlabel('Error (Real - Predicción)')
axes[1].set_title('Distribución de Errores')

plt.tight_layout()
plt.show()
```

**Asignaciones de datasets**

| Estudiante | Clasificación (fijo) | Regresión (fijo) | Dataset adicional asignado |
| :--- | :--- | :--- | :--- |
| Fernando Ballesteros | Diabetes | Precios de automóviles | SPAM |
| Dayanna Gallardo | Diabetes | Precios de automóviles | Análisis de Sentimientos |
| Maricela Gómez | Diabetes | Precios de automóviles | Titanic |
| Remigio Martínez | Diabetes | Precios de automóviles | Precios de Computadoras |
| Edgardo Sesma | Diabetes | Precios de automóviles | Peces |
| Carlos Zaragoza | Diabetes | Precios de automóviles | Social Media Post Likes |
| Elia Salas | Diabetes | Precios de automóviles | Calidad de Vino |
| Andrés Zavala | Diabetes | Precios de automóviles | Cáncer de mama |

## Precios de computadoras

El objetivo de este problema es construir un modelo para predecir el precio en dólares de computadoras portátiles basándose en sus especificaciones técnicas y componentes de hardware.

In [ ]:
!gdown 1whDUO7N6ztt4bEVDakhBQOOLCWtFxo_g

In [ ]:
import pandas as pd

computer_df = pd.read_csv('laptop_prices.csv')
computer_df

In [ ]:
#@title simplificación del dataset
import re

def clean_computer_data(df):
    df_clean = df.copy()

    # 1. Simplificar Almacenamiento (extraer capacidad en GB)
    def parse_storage(x):
        if pd.isna(x): return 0
        val = 0
        match = re.search(r'(\d+)\s*(GB|TB)', x, re.IGNORECASE)
        if match:
            amount = int(match.group(1))
            unit = match.group(2).upper()
            val = amount * 1024 if unit == 'TB' else amount
        return val

    df_clean['Storage_GB'] = df_clean['Storage'].apply(parse_storage)
    df_clean['Is_SSD'] = df_clean['Storage'].str.contains('SSD', case=False, na=False).astype(int)

    # 2. Simplificar GPU (Nvidia, AMD, Intel, Integrated)
    def simplify_gpu(x):
        if pd.isna(x): return 'Other'
        x_lower = x.lower()
        if 'nvidia' in x_lower: return 'Nvidia'
        elif 'amd' in x_lower or 'radeon' in x_lower: return 'AMD'
        elif 'intel' in x_lower: return 'Intel'
        elif 'integrated' in x_lower: return 'Integrated'
        return 'Other'

    df_clean['GPU_Brand'] = df_clean['GPU'].apply(simplify_gpu)

    # 3. Procesar Resolución de pantalla (calcular número total de píxeles)
    def parse_resolution(x):
        if pd.isna(x): return 0
        match = re.search(r'(\d+)\\s*x\\s*(\d+)', str(x), re.IGNORECASE)
        if match:
            return int(match.group(1)) * int(match.group(2))
        return 1920 * 1080 # valor por defecto si falla

    df_clean['Total_Pixels'] = df_clean['Resolution'].apply(parse_resolution)

    # Eliminar columnas originales que ya procesamos
    cols_to_drop = ['Storage', 'GPU', 'Resolution']
    df_clean = df_clean.drop(columns=cols_to_drop)

    return df_clean

computer_df = clean_computer_data(computer_df)
computer_df

## Titanic

El objetivo de esta sección consiste en desarrollar un modelo para predecir la probabilidad de supervivencia de los pasajeros a bordo del Titanic utilizando datos demográficos e información del viaje.

In [ ]:
!gdown 1mN6Av5IAiPS7QXYyuK0r5_CG0ILnb78d

In [ ]:
import pandas as pd

titanic_df = pd.read_csv('Titanic-Dataset.csv')
titanic_df

## Calidad del Vino



El objetivo de este problema es predecir la calidad del vino tinto a partir de sus propiedades químicas.

Más información: https://archive.ics.uci.edu/dataset/186/wine+quality

In [ ]:
import pandas as pd

url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
wine_df = pd.read_csv(url, sep=';')
wine_df

**¿Es tarea de clasificación o regresión?**

Argumenta y justifica cómo atacarás este problema.

In [ ]:
import matplotlib.pyplot as plt
from collections import Counter

countings = Counter(wine_df['quality'])
plt.bar(countings.keys(), countings.values())
plt.show()


In [ ]:
wine_df['quality'] = wine_df['quality'] - 3


## Precios de automóviles

El objetivo es predecir el precio de venta de automóviles a partir de características como el año, cilindraje, kilometraje y marca.

In [ ]:
!gdown 1XCzTrZ5HkiNHI3hjSTthJCOEhI5hZLOq

In [ ]:
import pandas as pd

car_df = pd.read_csv('car_price_dataset.csv')
car_df

## Diabetes

El objetivo de este ejercicio es predecir la presencia de diabetes en pacientes utilizando medidas de salud y variables fisiológicas.

Puedes retomar el preprocesamiento iniciado en la tarea 3.

In [ ]:
!gdown 1TM1voGUQ5NgNFyohSHazM8gPIOaFbiAt

In [ ]:
import pandas as pd

diabetes_df = pd.read_csv('diabetes.csv')
diabetes_df

## Social Media Likes

Crear un modelo capaz de predecir la cantidad aproximada de reacciones o 'likes' que recibirá una publicación en redes sociales a partir de su contenido textual.

In [ ]:
!gdown 12CkPYr94f-K8TyDFf5OiNtWS33eiR96U

In [ ]:
import pandas as pd

sm_df = pd.read_csv('post_likes.csv',index_col=0)
sm_df

## SPAM

El objetivo en este problema es clasificar mensajes de texto SMS entrantes si corresponden a correo no deseado (spam) o a mensajes legítimos (ham).

In [ ]:
!gdown 1-pMLSTkJ3ZPCKQU8oXA3uI-swvXw7DmJ

In [ ]:
import pandas as pd

spam_df = pd.read_csv('Spam_SMS.csv')
spam_df['Class'] = spam_df['Class'].map({'ham': 0, 'spam': 1})  # ¿Qué hace esta linea?
spam_df

## Peces

El objetivo de este ejercicio es entrenar un modelo que prediga la especie de un pez a partir de sus dimensiones corporales físicas y peso.

In [ ]:
!gdown 1VD19wYywiI3NbnXa5CZI21sAgc-0y2pp

In [ ]:
import pandas as pd

fish_df = pd.read_csv('fishes.csv')
fish_df

## Análisis de Sentimientos

El objetivo es predecir si el texto contiene una opinión positiva o negativa.

In [ ]:
!gdown 1YAqIVWIIPwNgHvB5lfwfVvsZlHWRCzmU

In [ ]:
import pandas as pd

as_df = pd.read_csv('rm21_balanced.csv')
as_df

🔴 ¿Puedes conjeturar de qué es el dataset?

## Cancer

El objetivo es clasificar tumores de mama en benignos o malignos utilizando métricas geométricas y características visuales obtenidas de imágenes digitalizadas de biopsias.

Más información: https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic

In [ ]:
!wget -O breast_cancer.zip https://archive.ics.uci.edu/static/public/17/breast+cancer+wisconsin+diagnostic.zip
!unzip -o breast_cancer.zip -d breast_cancer_data
!ls -l breast_cancer_data

In [ ]:
import pandas as pd

column_names = [
    'ID', 'Diagnosis',
    'radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean', 'concave_points_mean', 'symmetry_mean', 'fractal_dimension_mean',
    'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se', 'compactness_se', 'concavity_se', 'concave_points_se', 'symmetry_se', 'fractal_dimension_se',
    'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst', 'compactness_worst', 'concavity_worst', 'concave_points_worst', 'symmetry_worst', 'fractal_dimension_worst'
]

breast_cancer_df = pd.read_csv('breast_cancer_data/wdbc.data', header=None, names=column_names)
breast_cancer_df

# Parte 2

Agrega una celda de texto e investiga un concepto fundamental para la evaluación de modelos más allá de la clasificación binaria:

**1. Estrategias de Clasificación Multiclase**
Muchos algoritmos de Machine Learning son inherentemente binarios. Para aplicarlos a problemas con más de dos clases, la biblioteca implementa estrategias por debajo de la mesa. Investiga y explica:
* ¿En qué consiste la estrategia **One-vs-Rest** (OvR), también conocida habitualmente como **One-vs-All** (OvA)?
* ¿En qué consiste la estrategia alternativa **One-vs-One** (OvO)?
* Si tienes un problema con $K$ clases, ¿cuántos modelos binarios independientes entrena la estrategia OvR y cuántos entrena la estrategia OvO? ¿Cómo impacta esto en el costo computacional?

**2. Agregación de Métricas (Macro vs Micro)**
En problemas de clasificación multiclase, métricas como *Precision*, *Recall* y *F1-score* no se pueden calcular de forma directa global, sino que requieren estrategias de agregación. Las dos más utilizadas en bibliotecas como scikit-learn son **macro avg** y **micro avg**. Explica de manera concisa y rigurosa:
* **Definición operativa:** ¿Cuál es la diferencia matemática entre calcular el promedio *macro* y el promedio *micro*? (Escribe la formulación o explica detalladamente el orden de las operaciones en cada caso).
* **Aplicación práctica:** ¿Cómo afecta el desbalance de clases a cada una de estas métricas? ¿En qué escenario específico confiarías más en el *macro avg* y en cuál en el *micro avg*?

*Nota:* Estructura bien tu respuesta teórica; asegúrate de comprenderla a fondo, ya que estos conceptos serán fundamentales para futuras discusiones.